# LocalScript — QLoRA Fine-Tuning (MultiPLCoder-1b)

Fine-tunes `nuprl/MultiPLCoder-1b` (GPT-BigCode / StarCoder, 1B params) on
Octapi Lua generation tasks.

**Runtime:** T4 GPU (free tier) — model needs only ~4 GB VRAM.  
**Estimated time:** ~5 min on T4.  
**Dataset:** embedded — 314 examples, no upload needed.

In [ ]:
# Cell 1 — Install dependencies
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
    "unsloth", "trl", "transformers", "datasets",
    "peft", "bitsandbytes", "-q"], check=True)
print("✓ Dependencies installed.")

In [ ]:
# Cell 2 — Verify GPU
import torch
assert torch.cuda.is_available(), "No GPU — Runtime ▸ Change runtime type ▸ T4 GPU"
gpu  = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu}")
print(f"VRAM: {vram:.1f} GB")

In [ ]:
# Cell 3 — Decode embedded dataset (314 examples)
import base64, pathlib
DATASET_PATH = "train.jsonl"
pathlib.Path(DATASET_PATH).write_bytes(base64.b64decode(
    "..."
))
with open(DATASET_PATH) as f:
    n = sum(1 for line in f if line.strip())
print(f"✓ Dataset ready: {n} examples")

In [ ]:
# Cell 4 — Configuration
BASE_MODEL   = "nuprl/MultiPLCoder-1b"
MAX_SEQ_LEN  = 1024
OUTPUT_DIR   = "checkpoints"
LORA_R       = 16
EPOCHS       = 3
BATCH_SIZE   = 4
GRAD_ACCUM   = 2
LR           = 2e-4
LORA_TARGETS = ["c_attn", "c_proj", "c_fc"]  # GPT-BigCode
print(f"Model: {BASE_MODEL}  |  LoRA r={LORA_R}  |  Targets: {LORA_TARGETS}")

In [ ]:
# Cell 5 — Load model in 4-bit (Unsloth fast path, falls back to PEFT)
import torch

USE_UNSLOTH = False
try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True, dtype=None)
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_R, target_modules=LORA_TARGETS,
        lora_alpha=LORA_R, lora_dropout=0.0, bias="none",
        use_gradient_checkpointing="unsloth", random_state=42)
    USE_UNSLOTH = True
    print("✓ Model loaded via Unsloth.")
except Exception as exc:
    print(f"Unsloth unavailable ({exc!r}), using standard PEFT ...")
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_R,
        target_modules=LORA_TARGETS, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM"))
    print("✓ Model loaded via standard PEFT.")

In [ ]:
# Cell 6 — Tokenize dataset
# Plain-text completion: {system}\n\n{user}\n{output}<eos>
# Built from a Python list — avoids dill-pickling the Unsloth-patched tokenizer.
import json as _json
from datasets import Dataset

eos = tokenizer.eos_token or "<|endoftext|>"

with open(DATASET_PATH) as f:
    records = [_json.loads(line) for line in f if line.strip()]

all_ids = []
for r in records:
    inst, inp, out = r["instruction"], r.get("input",""), r["output"]
    text = f"{inst}\n\n{inp}\n{out}{eos}" if inp else f"{inst}\n{out}{eos}"
    ids = tokenizer(text, truncation=True, max_length=MAX_SEQ_LEN,
                    padding=False, return_attention_mask=False)["input_ids"]
    all_ids.append(ids)

dataset = Dataset.from_dict({"input_ids": all_ids, "labels": all_ids})
print(f"✓ {len(dataset)} examples tokenized.")

In [ ]:
# Cell 7 — Train
import os, json as _json, torch
from torch.nn.utils.rnn import pad_sequence
from transformers import Trainer, TrainingArguments

os.makedirs(OUTPUT_DIR, exist_ok=True)

pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id

def collate(batch):
    ids = pad_sequence([torch.tensor(ex["input_ids"], dtype=torch.long) for ex in batch],
                       batch_first=True, padding_value=pad_id)
    lbl = pad_sequence([torch.tensor(ex["labels"],    dtype=torch.long) for ex in batch],
                       batch_first=True, padding_value=-100)
    return {"input_ids": ids, "labels": lbl, "attention_mask": (ids != pad_id).long()}

trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    data_collator=collate,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        learning_rate=LR,
        fp16=False, bf16=True,
        logging_steps=10,
        save_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        report_to="none",
        seed=42,
        dataloader_num_workers=0,
    ),
)

print("Training started …")
stats = trainer.train()
final_dir = f"{OUTPUT_DIR}/final"
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
_json.dump({"base_model": BASE_MODEL, "use_unsloth": USE_UNSLOTH},
           open(f"{final_dir}/training_meta.json","w"), indent=2)
mins = stats.metrics["train_runtime"] / 60
print(f"\n✓ Done in {mins:.1f} min — checkpoint: {final_dir}")

In [ ]:
# Cell 8 — Merge LoRA + export to GGUF Q4_K_M
import os, glob, subprocess, sys, torch, json as _json

MERGED_DIR = "merged"
GGUF_PATH  = "localscript-q4_k_m.gguf"
final_dir  = f"{OUTPUT_DIR}/final"
meta       = _json.load(open(f"{final_dir}/training_meta.json"))

if meta["use_unsloth"]:
    from unsloth import FastLanguageModel
    m, t = FastLanguageModel.from_pretrained(final_dir, max_seq_length=MAX_SEQ_LEN,
                                              load_in_4bit=False, dtype=None)
    m.save_pretrained_gguf("localscript-gguf", t, quantization_method="q4_k_m")
    os.rename(glob.glob("localscript-gguf/*.gguf")[0], GGUF_PATH)
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(meta["base_model"],
               torch_dtype=torch.float16, device_map="cpu")
    PeftModel.from_pretrained(base, final_dir).merge_and_unload().save_pretrained(MERGED_DIR)
    AutoTokenizer.from_pretrained(meta["base_model"]).save_pretrained(MERGED_DIR)
    if not os.path.exists("llama.cpp"):
        subprocess.run(["git","clone","--depth","1",
            "https://github.com/ggerganov/llama.cpp.git"], check=True)
        subprocess.run([sys.executable,"-m","pip","install",
            "-r","llama.cpp/requirements.txt","-q"], check=True)
    subprocess.run([sys.executable,"llama.cpp/convert_hf_to_gguf.py",
        MERGED_DIR,"--outtype","q4_k_m","--outfile",GGUF_PATH], check=True)

print(f"✓ GGUF ready: {GGUF_PATH} ({os.path.getsize(GGUF_PATH)/1e6:.0f} MB)")

In [ ]:
# Cell 9 — Download
from google.colab import files
files.download(GGUF_PATH)

## Next Steps (local machine)

```bash
cp ~/Downloads/localscript-q4_k_m.gguf training/localscript-q4_k_m.gguf
cd training && ollama create localscript -f Modelfile
make eval
```